# 数据集导入

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('word2vec-nlp-tutorial')

print("Path to competition files:", path)



In [ ]:
# 解压数据
import glob
import os
import zipfile

# 压缩包所在的文件夹路径与解压目标路径
zip_dir = "/kaggle/input/competitions/word2vec-nlp-tutorial"  # 存放多个 zip 的文件夹
output_dir = "/kaggle/working/"  # 解压出来的目标文件夹

# 获取所有 .zip 文件的路径
zip_files = glob.glob(os.path.join(zip_dir, "*.zip"))

for file_path in zip_files:
    print(f"正在解压: {file_path}")
    with zipfile.ZipFile(file_path, "r") as zip_ref:
        # 解压到统一的目标文件夹（如果想按文件名单独建文件夹，可用下面的方法）
        zip_ref.extractall(output_dir)

print("所有 ZIP 文件解压完成！")

os.makedirs("./result", exist_ok=True)

## imdb-deberta-lora

In [ ]:
!pip install evaluate

In [8]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import time
import logging
import torch
import numpy as np
import pandas as pd
import datasets
import evaluate
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, DebertaV2Tokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType

program = os.path.basename(sys.argv[0]) if sys.argv and sys.argv[0] else "script"
logger = logging.getLogger(program)
logging.basicConfig(format="%(asctime)s: %(levelname)s: %(message)s", level=logging.INFO)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")

print("GPU:", torch.cuda.get_device_name(0))
print("Visible GPU count:", torch.cuda.device_count())
print("DataLoader workers: 2")

train_df = pd.read_csv("/kaggle/working/labeledTrainData.tsv", header=0, delimiter="\t", quoting=3)
test_df = pd.read_csv("/kaggle/working/testData.tsv", header=0, delimiter="\t", quoting=3)

class StepTimeCallback(TrainerCallback):
    def __init__(self, warmup_steps=10, measure_steps=100):
        self.warmup_steps = warmup_steps
        self.measure_steps = measure_steps
        self.step_start = None
        self.step_times = []
        self.total_start = None

    def on_step_begin(self, args, state, control, **kwargs):
        self.step_start = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self.step_start is None:
            return
        elapsed = time.perf_counter() - self.step_start
        step = state.global_step
        if step > self.warmup_steps and len(self.step_times) < self.measure_steps:
            self.step_times.append(elapsed)
        if len(self.step_times) == self.measure_steps:
            avg_time = sum(self.step_times) / len(self.step_times)
            print(f"\nBenchmark: {self.measure_steps} optimizer steps average = {avg_time:.3f} seconds")
            print(f"Estimated time for 1 epoch from optimizer steps: {avg_time * state.max_steps / max(1, args.num_train_epochs):.1f} seconds")
            self.step_times.append(float("inf"))

if __name__ == "__main__":
    train_df, val_df = train_test_split(
        train_df,
        test_size=0.2,
        random_state=42,
        stratify=train_df["sentiment"]
    )

    train_dataset = datasets.Dataset.from_dict({
        "label": train_df["sentiment"].astype(int).tolist(),
        "text": train_df["review"].tolist()
    })
    val_dataset = datasets.Dataset.from_dict({
        "label": val_df["sentiment"].astype(int).tolist(),
        "text": val_df["review"].tolist()
    })
    test_dataset = datasets.Dataset.from_dict({
        "text": test_df["review"].tolist()
    })

    model_id = "microsoft/deberta-v2-xxlarge"
    tokenizer = DebertaV2Tokenizer.from_pretrained(model_id)
    max_length = 128

    def preprocess_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_length)

    tokenized_train = train_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text"]
    )
    tokenized_val = val_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text"]
    )
    tokenized_test = test_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text"]
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query_proj", "value_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=["classifier"]
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        if isinstance(logits, tuple):
            logits = logits[0]
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    training_args = TrainingArguments(
        output_dir="./checkpoint",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        warmup_steps=125,
        weight_decay=0.01,
        logging_steps=25,
        save_strategy="no",
        eval_strategy="epoch",
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to="none",
        seed=42,
        data_seed=42
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[StepTimeCallback(warmup_steps=10, measure_steps=100)]
    )

    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype == torch.float16:
            param.data = param.data.float()
            if param.grad is not None:
                param.grad.data = param.grad.data.float()

    for name, param in model.named_parameters():
        if param.requires_grad:
            assert param.dtype != torch.float16, (
                f"FP16 trainable parameter remains: {name}"
            )

    print("All trainable parameters are FP32 or another non-FP16 dtype.")
    
    
    train_start = time.perf_counter()
    trainer.train()
    train_elapsed = time.perf_counter() - train_start

    print(f"Total training time: {train_elapsed / 60:.2f} minutes")

    prediction_outputs = trainer.predict(tokenized_test)
    test_pred = np.argmax(prediction_outputs.predictions, axis=-1).flatten()

    os.makedirs("./result", exist_ok=True)
    result_output = pd.DataFrame({
        "id": test_df["id"],
        "sentiment": test_pred
    })
    result_output.to_csv("./result/deberta_lora_workers2.csv", index=False, quoting=3)
    print("Result saved!")


GPU: Tesla T4
Visible GPU count: 1
DataLoader workers: 2


2026-10-18 06:52:35,590: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v2-xxlarge/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-18 06:52:35,606: INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v2-xxlarge/38f115b79f551ca4091c9caef41940fd1d6312b9/tokenizer_config.json "HTTP/1.1 200 OK"
2026-10-18 06:52:35,649: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v2-xxlarge/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-18 06:52:35,702: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v2-xxlarge/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-18 06:52:36,102: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v2-xxlarge "HTTP/1.1 200 OK"


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


All trainable parameters are FP32 or another non-FP16 dtype.


Epoch,Training Loss,Validation Loss,Accuracy
1,2.276549,0.303621,0.930000
2,3.309079,0.185223,0.934800



Benchmark: 100 optimizer steps average = 9.749 seconds
Estimated time for 1 epoch from optimizer steps: 12186.0 seconds
Total training time: 430.38 minutes


Result saved!


In [6]:
import torch
import peft
import accelerate
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
CUDA: True


In [ ]:
%pip install --no-cache-dir transformers==4.46.3 peft==0.13.2 accelerate==1.0.1